# Linear Interpolation Example

This notebook demonstrates how to use the PyFibreBundle package to remove the core pattern from fiber bundle images using triangular linear interpolation (TRILIN).

## Overview
- Load sample images (test image and calibration/background image)
- Create a PyBundle instance with TRILIN method
- Calibrate the bundle (one-time operation)
- Process the image to remove the core pattern
- Display and compare results

## Getting Started

Click Run -> Run All Cells

## Setup: Import Libraries

In [ ]:
import sys, time
from matplotlib import pyplot as plt
import numpy as np
from PIL import Image
from pathlib import Path

sys.path.append("../src")
from pybundle import PyBundle

## Load Images

We load two images:
1. **Test image**: The fiber bundle image we want to process
2. **Calibration/background image**: A uniform image used for normalisation and background reference

In [ ]:
# Load images from the test data directory
img = np.array(Image.open(Path('../test/data/usaf1.tif')))
calib_img = np.array(Image.open(Path('../test/data/usaf1_background.tif')))

print(f"Test image shape: {img.shape}")
print(f"Calibration image shape: {calib_img.shape}")
print(f"Test image dtype: {img.dtype}")

## Display Input Images

Let's visualize both the original test image and the calibration/background image.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5), dpi=100)

axes[0].imshow(img, cmap='gray')
axes[0].set_title('Original Test Image (with core pattern)')
axes[0].axis('off')

axes[1].imshow(calib_img, cmap='gray')
axes[1].set_title('Calibration/Background Image')
axes[1].axis('off')

plt.tight_layout()
plt.show()

## Create PyBundle Instance

Configure the PyBundle object with:
- **coreMethod**: Set to `TRILIN` for triangular linear interpolation
- **coreSize**: Estimated core spacing in pixels (helps identify core locations robustly)
- **calibImage**: Background/reference image for calibration
- **normaliseImage**: Image used for normalization

In [ ]:
# Create an instance of the PyBundle class 
pyb = PyBundle(
    coreMethod=PyBundle.TRILIN,        # Triangular linear interpolation method
    coreSize=3,                        # Estimated core spacing in pixels
    calibImage=calib_img,              # Background image for calibration
    normaliseImage=calib_img           # Image for normalization
)


## Calibration Step

The calibration step is computationally expensive (only done once). It:
- Detects core locations in the fiber bundle
- Sets up the triangulation for interpolation
- Prepares for the reconstruction process



In [ ]:
# Perform calibration
t1 = time.perf_counter()
pyb.calibrate()
calibration_time = time.perf_counter() - t1

print(f"Calibration completed in {round(1000 * calibration_time, 1)} ms")

## Process Image (First Call)

The first call to `process()` initializes the Numba JIT compiler. This call is slower than subsequent calls, so we do it now to get accurate timing information.

In [ ]:
# First call - initializes Numba JIT compiler
imgProc = pyb.process(img)
print("First processing completed (JIT compiled)")


## Process Image (Timed)

Now we process the image again and time it. This gives us the actual processing time without JIT compilation overhead.

In [ ]:
# Second call - actual processing timing
t1 = time.perf_counter()
imgProc = pyb.process(img)
processing_time = time.perf_counter() - t1

print(f"Reconstruction took {round(1000 * processing_time, 1)} ms")
print(f"Processed image shape: {imgProc.shape}")
print(f"Processed image dtype: {imgProc.dtype}")

## Display Results

Compare the original image (with core pattern) and the reconstructed image (core pattern removed).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5), dpi=100)

axes[0].imshow(img, cmap='gray')
axes[0].set_title('Original Image (with core pattern)')
axes[0].axis('off')

axes[1].imshow(imgProc, cmap='gray')
axes[1].set_title('Reconstructed Image (TRILIN - core pattern removed)')
axes[1].axis('off')

plt.tight_layout()
plt.show()